# 02 — Data Quality Investigation

## Objective

Investigate potential data quality issues identified during the initial audit.

This notebook focuses on:
- Customer-level metrics that differ from order-level aggregates.
- Potential outliers in product prices and order values.
- Missing values and their possible business meaning.
- Distinguishing genuine anomalies from valid business cases.

No records will be removed or modified without sufficient evidence.

In [1]:

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_DIR = Path("../data/raw")

datasets = {
    "products": pd.read_csv(RAW_DIR / "products.csv"),
    "customers": pd.read_csv(RAW_DIR / "customers.csv"),
    "orders": pd.read_csv(RAW_DIR / "orders.csv"),
    "order_items": pd.read_csv(RAW_DIR / "order_items.csv"),
    "reviews": pd.read_csv(RAW_DIR / "reviews.csv"),
}

products = datasets["products"]
customers = datasets["customers"]
orders = datasets["orders"]
order_items = datasets["order_items"]
reviews = datasets["reviews"]

print("All datasets loaded successfully!\n")

for name, df in datasets.items():
    print(f"{name}: {df.shape}")


All datasets loaded successfully!

products: (2000, 19)
customers: (800, 13)
orders: (3000, 21)
order_items: (6219, 10)
reviews: (2500, 10)


In [2]:

# INVESTIGATE CUSTOMER METRICS

tol = 0.01

delivered_orders = orders.loc[orders["status"] == "delivered"]

delivered_metrics = (
    delivered_orders.groupby("customer_id")
    .agg(
        calculated_orders_count=("id", "count"),
        calculated_total_spent=("total", "sum")
    )
    .reset_index()
)

customer_investigation = customers[
    ["id", "orders_count", "total_spent"]
].merge(
    delivered_metrics,
    left_on="id",
    right_on="customer_id",
    how="left"
)

customer_investigation["calculated_orders_count"] = (
    customer_investigation["calculated_orders_count"]
    .fillna(0)
    .astype(int)
)

customer_investigation["calculated_total_spent"] = (
    customer_investigation["calculated_total_spent"]
    .fillna(0)
)

customer_investigation["count_matches"] = (
    customer_investigation["orders_count"]
    == customer_investigation["calculated_orders_count"]
)

customer_investigation["spent_matches"] = (
    (
        customer_investigation["total_spent"]
        - customer_investigation["calculated_total_spent"]
    ).abs() <= tol
)

customer_investigation["both_match"] = (
    customer_investigation["count_matches"]
    & customer_investigation["spent_matches"]
)

print("CUSTOMER METRICS INVESTIGATION")
print("-" * 40)
print("Total customers:", len(customer_investigation))
print("Both metrics match:", customer_investigation["both_match"].sum())
print("Count mismatches:", (~customer_investigation["count_matches"]).sum())
print("Spending mismatches:", (~customer_investigation["spent_matches"]).sum())

display(
    customer_investigation.loc[
        ~customer_investigation["both_match"],
        [
            "id",
            "orders_count",
            "calculated_orders_count",
            "total_spent",
            "calculated_total_spent",
            "count_matches",
            "spent_matches"
        ]
    ].head(15).to_string(index=False)
)


CUSTOMER METRICS INVESTIGATION
----------------------------------------
Total customers: 800
Both metrics match: 753
Count mismatches: 47
Spending mismatches: 47


' id  orders_count  calculated_orders_count  total_spent  calculated_total_spent  count_matches  spent_matches\n 27             8                        7     18664.56                17788.00          False          False\n 39             6                        5     15482.64                14216.91          False          False\n 41             4                        3     13097.50                12354.77          False          False\n 62             5                        4      4873.13                 4201.80          False          False\n 72             4                        3      1585.22                 1123.78          False          False\n 79             2                        1      5426.87                 1737.69          False          False\n123             2                        1      1229.22                  461.87          False          False\n127             6                        5      5543.32                 3266.24          False          False\n

In [3]:

# COMPARE CUSTOMER METRICS ACROSS ORDER STATUS RULES

all_metrics = (
    orders.groupby("customer_id")
    .agg(
        all_orders_count=("id", "count"),
        all_orders_spent=("total", "sum")
    )
    .reset_index()
)

non_cancelled_orders = orders.loc[orders["status"] != "cancelled"]

non_cancelled_metrics = (
    non_cancelled_orders.groupby("customer_id")
    .agg(
        non_cancelled_count=("id", "count"),
        non_cancelled_spent=("total", "sum")
    )
    .reset_index()
)

status_comparison = (
    customer_investigation
    .merge(all_metrics, left_on="id", right_on="customer_id", how="left")
    .merge(non_cancelled_metrics, left_on="id", right_on="customer_id", how="left")
)

status_comparison["matches_all_orders"] = (
    status_comparison["orders_count"].eq(status_comparison["all_orders_count"])
    & (
        status_comparison["total_spent"]
        - status_comparison["all_orders_spent"]
    ).abs().le(tol)
)

status_comparison["matches_non_cancelled"] = (
    status_comparison["orders_count"].eq(status_comparison["non_cancelled_count"])
    & (
        status_comparison["total_spent"]
        - status_comparison["non_cancelled_spent"]
    ).abs().le(tol)
)

status_comparison["matches_delivered"] = status_comparison["both_match"]

print("MATCH COUNTS ACROSS POSSIBLE DEFINITIONS")
print("All orders:", status_comparison["matches_all_orders"].sum())
print("Non-cancelled orders:", status_comparison["matches_non_cancelled"].sum())
print("Delivered orders:", status_comparison["matches_delivered"].sum())

mismatches = status_comparison.loc[
    ~status_comparison["both_match"],
    [
        "id",
        "orders_count",
        "total_spent",
        "all_orders_count",
        "all_orders_spent",
        "non_cancelled_count",
        "non_cancelled_spent",
        "calculated_orders_count",
        "calculated_total_spent",
        "matches_all_orders",
        "matches_non_cancelled"
    ]
]

display(mismatches.head(15).to_string(index=False))


MATCH COUNTS ACROSS POSSIBLE DEFINITIONS
All orders: 543
Non-cancelled orders: 677
Delivered orders: 753


' id  orders_count  total_spent  all_orders_count  all_orders_spent  non_cancelled_count  non_cancelled_spent  calculated_orders_count  calculated_total_spent  matches_all_orders  matches_non_cancelled\n 27             8     18664.56               8.0          18664.56                  8.0             18664.56                        7                17788.00                True                   True\n 39             6     15482.64               6.0          15482.64                  6.0             15482.64                        5                14216.91                True                   True\n 41             4     13097.50               4.0          13097.50                  4.0             13097.50                        3                12354.77                True                   True\n 62             5      4873.13               5.0           4873.13                  5.0              4873.13                        4                 4201.80                True              

In [4]:

# CLASSIFY CUSTOMER METRIC MISMATCHES

def classify_customer(row):
    if row["matches_delivered"]:
        return "Matches delivered"
    elif row["matches_non_cancelled"]:
        return "Matches non-cancelled only"
    elif row["matches_all_orders"]:
        return "Matches all orders only"
    else:
        return "Matches none of the tested rules"

status_comparison["metric_classification"] = status_comparison.apply(
    classify_customer, axis=1
)

print("CUSTOMER METRIC CLASSIFICATION")
print("-" * 40)

print(
    status_comparison["metric_classification"]
    .value_counts()
    .to_string()
)

print("\nCustomers matching none of the tested rules:")

display(
    status_comparison.loc[
        status_comparison["metric_classification"]
        == "Matches none of the tested rules",
        [
            "id",
            "orders_count",
            "total_spent",
            "all_orders_count",
            "all_orders_spent",
            "non_cancelled_count",
            "non_cancelled_spent",
            "calculated_orders_count",
            "calculated_total_spent"
        ]
    ].to_string(index=False)
)


CUSTOMER METRIC CLASSIFICATION
----------------------------------------
metric_classification
Matches delivered                   753
Matches non-cancelled only           43
Matches none of the tested rules      4

Customers matching none of the tested rules:


' id  orders_count  total_spent  all_orders_count  all_orders_spent  non_cancelled_count  non_cancelled_spent  calculated_orders_count  calculated_total_spent\n169             3      3285.58               5.0           6007.93                  5.0              6007.93                        2                 2516.59\n214             4      7409.11               5.0           7596.66                  5.0              7596.66                        3                 5826.56\n455             5      2897.91               6.0           3062.12                  6.0              3062.12                        4                 1946.96\n543             5      5422.95               6.0           5833.60                  6.0              5833.60                        4                 3560.50'

### Investigation 1 — Customer Metrics Consistency

The `customers` dataset contains two aggregate metrics: `orders_count` and `total_spent`. These were compared with values calculated from the `orders` dataset under three possible definitions: delivered orders, non-cancelled orders, and all orders.

**Findings:**
- 753 of 800 customers (94.1%) matched both metrics calculated from delivered orders.
- 43 customers matched both metrics calculated from non-cancelled orders but not delivered orders.
- 4 customers (IDs 169, 214, 455, and 543) matched none of the tested definitions.

The results suggest that customer-level metrics may use different order-status rules. However, the intended source-system definition has not been confirmed.

**Decision:** Do not overwrite the customer metrics or remove these records. Retain the four unmatched customers for further investigation and document the uncertainty before downstream analysis.

In [5]:

# INVESTIGATE THE HIGHEST-VALUE ORDERS

top_order_ids = orders.nlargest(10, "total")["id"]

top_order_details = (
    order_items.loc[order_items["order_id"].isin(top_order_ids)]
    .merge(
        orders[["id", "status", "total"]],
        left_on="order_id",
        right_on="id",
        suffixes=("_item", "_order")
    )
    .merge(
        products[["id", "name", "category_id", "price"]],
        left_on="product_id",
        right_on="id",
        suffixes=("", "_product")
    )
)

display(
    top_order_details[
        [
            "order_id",
            "status",
            "total",
            "name",
            "category_id",
            "quantity",
            "unit_price",
            "price",
            "line_total"
        ]
    ]
    .sort_values(["total", "order_id"], ascending=[False, True])
    .round(2)
    .to_string(index=False)
)


' order_id    status    total                                     name  category_id  quantity  unit_price   price  line_total\n     1043   shipped 17154.90            Granite Smart Jump Rope Olive            5         2      196.82  196.82      393.64\n     1043   shipped 17154.90          Northlight Gaming Desktop Olive            2         4     2842.61 2842.61    11370.44\n     1043   shipped 17154.90          Hearth & Co. Scented Lamp Ivory            3         4      233.31  233.31      933.24\n     1043   shipped 17154.90          Motive Foldable Kettlebell Navy            5         3     1109.90 1109.90     3329.70\n      840 delivered 16857.37            Trailhead Alpine Sleeping Bag            4         1      367.45  367.45      367.45\n      840 delivered 16857.37       Hearth & Co. Ceramic Vase Graphite            3         1      183.29  183.29      183.29\n      840 delivered 16857.37             Quill Rugged Docking Station            2         4     3492.50 3492.50    1

In [6]:

# PRODUCT PRICE OUTLIERS BY CATEGORY

q1 = products["price"].quantile(0.25)
q3 = products["price"].quantile(0.75)
iqr = q3 - q1

upper_bound = q3 + 1.5 * iqr

products_outlier_check = products[
    ["id", "name", "category_id", "price", "status"]
].copy()

products_outlier_check["is_price_outlier"] = (
    products_outlier_check["price"] > upper_bound
)

category_outlier_summary = (
    products_outlier_check.groupby("category_id")
    .agg(
        product_count=("id", "count"),
        outlier_count=("is_price_outlier", "sum"),
        median_price=("price", "median"),
        max_price=("price", "max")
    )
    .reset_index()
)

category_outlier_summary["outlier_percentage"] = (
    category_outlier_summary["outlier_count"]
    / category_outlier_summary["product_count"]
    * 100
)

print(f"IQR upper bound: ${upper_bound:.2f}")
display(
    category_outlier_summary
    .sort_values("outlier_percentage", ascending=False)
    .round(2)
    .to_string(index=False)
)


IQR upper bound: $1268.25


' category_id  product_count  outlier_count  median_price  max_price  outlier_percentage\n           2            260            159       1553.16    3492.50               61.15\n           5            261              4        547.65    1296.63                1.53\n           3            260              0        179.08     348.83                0.00\n           1            244              0        433.86     895.87                0.00\n           4            246              0        407.88     797.38                0.00\n           6            251              0         52.76      88.93                0.00\n           7            225              0         99.46     198.44                0.00\n           8            253              0        218.77     448.14                0.00'

In [7]:

# CATEGORY-SPECIFIC IQR OUTLIER ANALYSIS

category_iqr_results = []

for category_id, group in products.groupby("category_id"):
    q1 = group["price"].quantile(0.25)
    q3 = group["price"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (group["price"] < lower_bound)
        | (group["price"] > upper_bound)
    )

    category_iqr_results.append({
        "category_id": category_id,
        "product_count": len(group),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": int(outliers.sum()),
        "outlier_percentage": outliers.mean() * 100,
        "median_price": group["price"].median(),
        "max_price": group["price"].max()
    })

category_iqr_summary = pd.DataFrame(category_iqr_results)

display(
    category_iqr_summary
    .sort_values("category_id")
    .round(2)
    .to_string(index=False)
)


' category_id  product_count  lower_bound  upper_bound  outlier_count  outlier_percentage  median_price  max_price\n           1            244      -376.77      1256.91              0                 0.0        433.86     895.87\n           2            260     -1592.47      4985.00              0                 0.0       1553.16    3492.50\n           3            260      -140.27       516.78              0                 0.0        179.08     348.83\n           4            246      -350.45      1162.02              0                 0.0        407.88     797.38\n           5            261      -772.05      1939.87              0                 0.0        547.65    1296.63\n           6            251       -22.30       124.48              0                 0.0         52.76      88.93\n           7            225       -74.68       287.88              0                 0.0         99.46     198.44\n           8            253      -221.64       657.96              0           

### Investigation 2 — Product Price Outliers

The global IQR method identified 163 products (8.15%) as potential price outliers using an upper bound of $1,268.25.

A category-level IQR analysis was then performed because product categories may have substantially different price distributions.

**Findings:**
- Category 2 contained 159 products above the global IQR upper bound, representing 61.15% of that category.
- Category-level IQR analysis identified no outliers in any of the eight categories.
- The results suggest that the global IQR threshold was influenced by differences in typical prices across categories.

**Decision:** Do not remove or alter products based solely on the global IQR results. Retain the observed prices and use category-aware analysis when investigating price anomalies. The absence of category-level IQR outliers does not rule out every possible data quality issue.

In [8]:

# ORDER VALUE DISTRIBUTION BY STATUS

order_status_summary = (
    orders.groupby("status")
    .agg(
        order_count=("id", "count"),
        median_total=("total", "median"),
        mean_total=("total", "mean"),
        max_total=("total", "max"),
        median_subtotal=("subtotal", "median"),
        max_subtotal=("subtotal", "max")
    )
    .reset_index()
)

display(
    order_status_summary
    .sort_values("median_total", ascending=False)
    .round(2)
    .to_string(index=False)
)


'   status  order_count  median_total  mean_total  max_total  median_subtotal  max_subtotal\n  shipped           32       1389.72     2153.89   17154.90          1289.01      16027.02\n     paid            9        950.95     1009.57    1784.70           883.14       1667.94\n  pending            7        849.77     1202.22    3689.18           780.17       3433.82\ncancelled          173        802.63     1590.29   15545.57           736.11      14522.97\ndelivered         2675        790.20     1419.85   16857.37           744.12      15748.95\n refunded          104        639.18     1159.96    7144.77           605.55       6671.76'

### Investigation 3 — Order Values by Status

Order totals were compared across order statuses to investigate whether unusually high values might indicate data quality issues.

**Findings:**
- The highest order total was $17,154.90, belonging to a shipped order.
- High-value orders also appeared among cancelled and delivered orders.
- The mean order total exceeded the median in every status group, indicating that larger orders influenced the average.
- The number of records differed considerably between status groups, so comparisons involving smaller groups should be interpreted cautiously.

**Decision:** No orders were removed or modified based on their monetary values. Order status will be considered in subsequent revenue and customer behaviour analyses, particularly when distinguishing completed purchases from cancelled or refunded orders.